In [1]:
import sys
sys.path.append('../')

In [2]:
import pickle as pkl
import json

import numpy as np
import heapq
from copy import deepcopy
from tqdm import tqdm

In [3]:
from src.envs.trainer import Trainer, WOOD_MAZES, BRONZE_MAZES
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS

from src.envs.agents.agent_factory import get_agent
from src.envs.experiments import get_top_k, get_agent_info, _get_agent_info, get_trainer, play_round
from src.envs.agents.agent_factory import get_agent
from src.envs.league.elo_league import EloLeague
from src.envs.league.agent_description import AgentDescription
from src.envs.league.league import League

In [4]:
with open('dqn_exp.pkl', 'rb') as f:
    data = pkl.load(f)

In [5]:
agent_descs = [
    AgentDescription(
        exp_name=v['exp_name'],
        best_iter=v['best_iter'],
        agent_id=0,
    )
    for k,v in data.items()
]

In [6]:
with open('league.pkl', 'rb') as f:
    elo_league = pkl.load(f)

In [7]:
league = League(main_agent=None, fixed_agents_desc=agent_descs)
league.elo_league = elo_league

In [8]:
best_agent = league.find_best()

In [9]:
best_params = data[best_agent.exp_name]

In [10]:
best_agent.agent_info

{'train': False,
 'type': 'qdn_conv',
 'action_space_n': 8,
 'state_type': 'conv',
 'batch_size': 32,
 'prioritized_replay': True,
 'buffer_params': {'alpha': 0.3174810314736277,
  'beta': 0.9610239806310656,
  'need_aug': True,
  'capacity': 1000},
 'model_params': {'fc_dim': 16, 'conv_dim': 8},
 'sync_target_frames': 4000,
 'replay_start_size': 1000,
 'epsilon_params': {'start': 1.0,
  'final': 0.05,
  'decay': 400000,
  'reset': 15000,
  'reset_coef': 1},
 'gamma': 0.9593589973534757,
 'size': 4,
 'lr': 0.0001,
 'optimizer': 'adamw',
 'scheduler_params': {'type': 'cosine', 'T_max': 800},
 'checkpoint_dir': '../output/2025-06-21/20250621-220055/agent0/300'}

In [11]:
# agent = best_agent.agent_info
# def train_exploiter(self, agent, exploiter):

In [12]:
# from src.envs.experiments import get_trainer

In [13]:
LEAGUE_LEVEL = 3
NUM_EXPERIMENTS = 1000
NUM_ENVS = 4
random_epsilon = 0.5

In [14]:
DEFAULT_EXPLOITER = AgentDescription(
    exp_name='20250629-174451',
    best_iter=3000,
    agent_id=0,
)

In [15]:
expoiter_agent_info = deepcopy(DEFAULT_EXPLOITER.agent_info)
expoiter_agent_info['train'] = True

agent_info = deepcopy(best_agent.agent_info)
agent_info['train'] = False

# expoiter_agent_info['checkpoint_dir'] = agent_info['checkpoint_dir']
# expoiter_agent_info['drop_layers'] = ["fc2.weight", "fc2.bias"]

In [16]:
expoiter_agent_info['reward_params']['other_reward_coef'] = 0.1
# expoiter_agent_info['reward_params']['good_explorers_nearby_bonus'] = 0.1
# expoiter_agent_info['reward_params']['bad_explorers_nearby_bonus'] = 0

In [17]:
agents_info = [
    expoiter_agent_info,
    agent_info,
]

In [18]:
env_kwargs={
    'reward_params': {
        # 'sanity_coef': best_params['params_sanity_coef'],
        # 'reward_for_win': best_params['params_reward_for_win'],
        # 'reward_for_lose': best_params['params_reward_for_lose'],
        'sanity_coef': 0.1,
        'reward_for_win': 0,
        'reward_for_lose': -2,
    }
}

In [19]:
env_kwargs

{'reward_params': {'sanity_coef': 0.1,
  'reward_for_win': 0,
  'reward_for_lose': -2}}

In [48]:
agents_info

[{'train': True,
  'type': 'ppo',
  'action_space_n': 8,
  'actions': ['UP', 'RIGHT', 'DOWN', 'LEFT', 'WAIT', 'PLAN', 'LIGHT', 'YELL'],
  'state_type': 'conv',
  'model_params': {'fc_dim': 16, 'conv_dim': 8, 'size': 4},
  'gamma': 0.95,
  'lr': 0.0001,
  'optimizer': 'adamw',
  'scheduler_params': {'type': 'cosine', 'T_max': 3000},
  'entropy_coef': 0.27,
  'value_loss_coef': 0.26,
  'clip_ratio': 0.05,
  'ppo_epochs': 5,
  'mini_batch_size': 16,
  'target_kl': 0.09,
  'max_grad_norm': 0.16,
  'gae_lambda': 0.91,
  'reward_params': {'good_plan_bonus': 0.05,
   'bad_plan_bonus': -0.1,
   'good_light_bonus': 0.1,
   'bad_light_bonus': -0.1,
   'other_reward_coef': 0.1},
  'checkpoint_dir': '../output/2025-06-29/20250629-174451/agent0/3000'},
 {'train': False,
  'type': 'qdn_conv',
  'action_space_n': 8,
  'state_type': 'conv',
  'batch_size': 32,
  'prioritized_replay': True,
  'buffer_params': {'alpha': 0.3174810314736277,
   'beta': 0.9610239806310656,
   'need_aug': True,
   'capacity

In [20]:
trainer = Trainer(
    num_experiments=NUM_EXPERIMENTS, agents_info=agents_info,
    league_level=LEAGUE_LEVEL, actions=EXTENDED_KUTULU_ACTIONS, mazes=BRONZE_MAZES, log_dir='../runs',
    num_envs=NUM_ENVS, env_kwargs=env_kwargs
)

In [21]:
result = trainer.train()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [18:31<00:00,  1.11s/it]


In [53]:
scores = np.array([[np.argwhere(~np.isnan(r[:,i])).max().item() for i in range(2)] for r in result[0]])

In [55]:
scores

array([[60, 60],
       [66, 66],
       [60, 60],
       ...,
       [76, 76],
       [74, 74],
       [83, 83]], shape=(1000, 2))

In [58]:
trainer.log_dir.split('/')[-1]

'20250706-173214'

In [17]:
# agent = get_agent(deepcopy(expoiter_agent_info))
# random_agent_info = {
#     'type': 'epsilon_wait',
#     'action_space_n': len(EXTENDED_KUTULU_ACTIONS),
#     'epsilon_params': {'start': random_epsilon, 'final': random_epsilon, 'decay': int(4 * 10**5)},
#     'state_type': 'closest',
#     'action': 'WAIT',
# }
# agents_info += [
#     deepcopy(random_agent_info),
#     deepcopy(random_agent_info),
# ]

In [18]:
# trainer = Trainer(
#     num_experiments=1, agents_info=agents_info,
#     league_level=LEAGUE_LEVEL, actions=EXTENDED_KUTULU_ACTIONS, mazes=BRONZE_MAZES, log_dir='../runs',
#     num_envs=NUM_ENVS, verbose=True
# )

In [19]:
# _ = trainer.train()

In [24]:
from typing import List, Dict
def play_round(agent_descs: List[AgentDescription],
               num_experiments=1, league_level=3, num_envs=1):
    n_agents = len(agent_descs)
    # assert len(agent_descs) == 4
    agents_info = [
        a.agent_info
        for a in agent_descs
    ]
    mazes = BRONZE_MAZES if league_level >= 3 else WOOD_MAZES
    actions = EXTENDED_KUTULU_ACTIONS if league_level >= 3 else DEFAULT_KUTULU_ACTIONS
    trainer = Trainer(
        num_experiments=num_experiments, agents_info=agents_info, shuffle=True,
        league_level=league_level, mazes=mazes, actions=actions, log_dir='../runs', verbose=False,
        silent=True, num_envs=num_envs, only_train=False, use_tqdm=False,
    )
    result = trainer.train()
    scores = np.array([[np.argwhere(~np.isnan(r[:,i])).max().item() for i in range(n_agents)] for r in result[0]])
    return scores

In [25]:
agent_descs_1 = [
    best_agent,
    DEFAULT_EXPLOITER,
]

In [26]:
scores_1 = play_round(agent_descs_1, num_experiments=10)
scores_1

array([[90, 63],
       [60, 57],
       [81, 82],
       [74, 56],
       [89, 49],
       [73, 72],
       [66, 66],
       [84, 48],
       [41, 60],
       [47, 70]])

In [27]:
agent_descs_2 = [
    best_agent,
    AgentDescription(
        exp_name='20250705-220950',
        best_iter=1000,
        agent_id=0,
    ),
]

In [28]:
scores_2 = play_round(agent_descs_2, num_experiments=10)
scores_2

array([[68, 40],
       [62, 64],
       [43, 42],
       [45, 42],
       [79, 69],
       [75, 37],
       [48, 45],
       [64, 62],
       [67, 46],
       [43, 40]])

In [29]:
agent_descs_3 = [
    best_agent,
    AgentDescription(
        exp_name='20250706-152304',
        best_iter=900,
        agent_id=0,
    ),
]

In [30]:
scores_3 = play_round(agent_descs_3, num_experiments=10)
scores_3

array([[61, 46],
       [88, 46],
       [42, 40],
       [60, 46],
       [44, 42],
       [56, 61],
       [50, 57],
       [55, 66],
       [45, 43],
       [69, 45]])

In [31]:
agent_descs_4 = [
    best_agent,
    AgentDescription(
        exp_name='20250706-153823',
        best_iter=1000,
        agent_id=0,
    ),
]

In [32]:
scores_4 = play_round(agent_descs_4, num_experiments=10)
scores_4

array([[63, 64],
       [67, 65],
       [60, 58],
       [66, 65],
       [40, 38],
       [80, 62],
       [42, 69],
       [62, 60],
       [61, 68],
       [44, 50]])

In [33]:
agent_descs_5 = [
    best_agent,
    AgentDescription(
        exp_name='20250706-173214',
        best_iter=1000,
        agent_id=0,
    ),
]

In [46]:
agent_descs_5[0].agent_info

{'train': False,
 'type': 'qdn_conv',
 'action_space_n': 8,
 'state_type': 'conv',
 'batch_size': 32,
 'prioritized_replay': True,
 'buffer_params': {'alpha': 0.3174810314736277,
  'beta': 0.9610239806310656,
  'need_aug': True,
  'capacity': 1000},
 'model_params': {'fc_dim': 16, 'conv_dim': 8},
 'sync_target_frames': 4000,
 'replay_start_size': 1000,
 'epsilon_params': {'start': 1.0,
  'final': 0.05,
  'decay': 400000,
  'reset': 15000,
  'reset_coef': 1},
 'gamma': 0.9593589973534757,
 'size': 4,
 'lr': 0.0001,
 'optimizer': 'adamw',
 'scheduler_params': {'type': 'cosine', 'T_max': 800},
 'checkpoint_dir': '../output/2025-06-21/20250621-220055/agent0/300'}

In [47]:
agent_descs_5[1].agent_info

{'train': False,
 'type': 'ppo',
 'action_space_n': 8,
 'actions': ['UP', 'RIGHT', 'DOWN', 'LEFT', 'WAIT', 'PLAN', 'LIGHT', 'YELL'],
 'state_type': 'conv',
 'model_params': {'fc_dim': 16, 'conv_dim': 8, 'size': 4},
 'gamma': 0.95,
 'lr': 0.0001,
 'optimizer': 'adamw',
 'scheduler_params': {'type': 'cosine', 'T_max': 3000},
 'entropy_coef': 0.27,
 'value_loss_coef': 0.26,
 'clip_ratio': 0.05,
 'ppo_epochs': 5,
 'mini_batch_size': 16,
 'target_kl': 0.09,
 'max_grad_norm': 0.16,
 'gae_lambda': 0.91,
 'reward_params': {'good_plan_bonus': 0.05,
  'bad_plan_bonus': -0.1,
  'good_light_bonus': 0.1,
  'bad_light_bonus': -0.1,
  'other_reward_coef': 0.1},
 'checkpoint_dir': '../output/2025-07-06/20250706-173214/agent0/1000'}

In [34]:
scores_5 = play_round(agent_descs_5, num_experiments=10)
scores_5

array([[87, 84],
       [76, 71],
       [65, 66],
       [66, 53],
       [55, 57],
       [74, 73],
       [57, 81],
       [66, 68],
       [63, 81],
       [85, 55]])

In [37]:
agent_descs_6 = [
    best_agent,
    AgentDescription(
        exp_name='20250706-171057',
        best_iter=500,
        agent_id=0,
    ),
]

In [38]:
scores_6 = play_round(agent_descs_6, num_experiments=10)
scores_6

array([[61, 60],
       [55, 52],
       [75, 87],
       [63, 63],
       [59, 58],
       [50, 59],
       [65, 52],
       [63, 53],
       [60, 57],
       [74, 72]])

In [39]:
scores_1.argmax(axis=1)

array([0, 0, 1, 0, 0, 0, 0, 0, 1, 1])

In [40]:
scores_2.argmax(axis=1)

array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0])

In [41]:
scores_3.argmax(axis=1)

array([0, 0, 0, 0, 0, 1, 1, 1, 0, 0])

In [42]:
scores_4.argmax(axis=1)

array([1, 0, 0, 0, 0, 0, 1, 0, 1, 1])

In [43]:
scores_5.argmax(axis=1)

array([0, 0, 1, 0, 1, 0, 1, 1, 1, 0])

In [44]:
scores_6.argmax(axis=1)

array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0])

In [35]:
rewards = [11.2,None,13.2,14.1]

In [36]:
cur_rewards = list(rewards)
cur_rewards.pop(2)

13.2

In [37]:
min(cur_rewards)

TypeError: '<' not supported between instances of 'NoneType' and 'float'

In [26]:
cur_rewards, rewards

([11, None, 14], [11, None, 13, 14])